# Collider Bias: Selection Changes the Evidence

## A dataset is not just data — it is the result of a selection process

## Takeaway

Functions introduced: `query`, `loc`, `iloc`.

**Concept learned: selection can invent or hide relationships.**

Run the cell below first. It enlarges the font for both code and markdown so the notebook is easy to read while walking through it in class.

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<style>
/* Rendered markdown */
.jp-RenderedHTMLCommon,
.jp-RenderedMarkdown,
.rendered_html {
    font-size: 24px !important;
    line-height: 1.5 !important;
}
.jp-RenderedHTMLCommon h1, .rendered_html h1 { font-size: 40px !important; }
.jp-RenderedHTMLCommon h2, .rendered_html h2 { font-size: 34px !important; }
.jp-RenderedHTMLCommon h3, .rendered_html h3 { font-size: 30px !important; }
.jp-RenderedHTMLCommon h4, .rendered_html h4 { font-size: 28px !important; }
.jp-RenderedHTMLCommon table, .rendered_html table {
    font-size: 22px !important;
}

/* Code editor (CodeMirror, used by classic + JupyterLab) */
.CodeMirror, .cm-editor, .jp-Editor, .jp-InputArea-editor {
    font-size: 24px !important;
}
.cm-content, .cm-line { font-size: 24px !important; }

/* Code output (print, tracebacks, DataFrame text) */
.jp-OutputArea-output,
.output_area,
.output pre,
.jp-RenderedText pre {
    font-size: 22px !important;
}

/* DataFrame tables in output */
.dataframe, .dataframe th, .dataframe td {
    font-size: 22px !important;
}
</style>
"""))


### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (8, 5)

## The story

Applicants are selected using ability **and** portfolio. Inside admitted students, the relationship between those two variables can look very different — even reversed — compared to the full applicant pool. The doorway distorts the evidence.

## 1. Load the admissions data

Open the file and ask whether it is the full pool or a selected subset.

In [ ]:
apps = pd.read_csv("../data/collider_admissions.csv")
apps.head()

In [ ]:
apps.info()

## 2. Find the selection clue

A source or status column often reveals how rows entered the table.

In [ ]:
apps["data_source"].value_counts()

In [ ]:
apps["admitted"].value_counts()

## 3. Filter with `df.query()`

`query()` is the most readable way to express many row-selection stories.

In [ ]:
admitted = apps.query("admitted == True")
not_admitted = apps.query("admitted == False")
admitted.shape, not_admitted.shape

## 4. Compare full vs. selected

Now look at the correlation between ability and portfolio in the full pool, then inside the admitted subset.

In [ ]:
apps[["ability", "portfolio_score"]].corr()

In [ ]:
admitted[["ability", "portfolio_score"]].corr()

Notice the correlation flips sign or weakens dramatically. That is collider bias in action — conditioning on `admitted` induced a relationship that was not there in the full pool.

## 5. Visualize it

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
apps.plot(kind="scatter", x="ability", y="portfolio_score",
          ax=axes[0], title="All applicants", alpha=0.4)
admitted.plot(kind="scatter", x="ability", y="portfolio_score",
              ax=axes[1], title="Admitted only", alpha=0.4, color="C1")
plt.tight_layout()
plt.show()

## 6. Compound conditions

Real selection rules often combine multiple conditions.

In [ ]:
apps.query("admitted == True and ability > 0").shape

In [ ]:
apps.query("portfolio_score > 1 or ability > 1").shape

## 7. Column selection with `df.loc[]`

`loc` selects by labels: rows by condition, columns by name.

In [ ]:
apps.loc[:, ["ability", "portfolio_score", "admitted"]].head()

## 8. Rows and columns together with `loc`

In [ ]:
apps.loc[apps["admitted"] == True, ["ability", "portfolio_score"]].head()

## 9. Position selection with `df.iloc[]`

`iloc` is for quick position-based checks.

In [ ]:
apps.iloc[:5, :4]

## Mini-lab: recreate the bias

Compute correlations before and after selection.

In [ ]:
full_corr = apps[["ability", "portfolio_score"]].corr()
selected_corr = apps.query("admitted == True")[["ability", "portfolio_score"]].corr()
print("Full pool correlation:")
print(full_corr)
print("\nAdmitted-only correlation:")
print(selected_corr)

## Collection questions

For every dataset, ask:

- Who was eligible?
- Who was actually measured?
- Who is absent?
- Who had to pass through a doorway to be here?

Real-world colliders: hospitals, elite schools, customer support tickets, dating apps, product reviews, job interviews.